In [0]:
# ==============================================================================
# BLOCO 1: INSTALAÇÃO DE DEPENDÊNCIAS E IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================

%pip install openpyxl plotly

import pandas as pd
import numpy as np
import os
import re
import plotly.express as px
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import CountVectorizer

print("Bibliotecas importadas com sucesso!")

In [0]:
# ==============================================================================
# BLOCO 2: CARREGAMENTO DA BASE DE DADOS
# ==============================================================================


# Caminho do diretório (volume) no Databricks
diretorio = "/Volumes/workspace/default/mestrado_analisedoprototipo/"

# Localiza o arquivo Excel automaticamente
arquivos_no_volume = os.listdir(diretorio)
ficheiro_excel = [f for f in arquivos_no_volume if f.endswith('.xlsx')][0]
caminho_arquivo = os.path.join(diretorio, ficheiro_excel)

print(f"Arquivo localizado: {ficheiro_excel}")

# Carregamento do DataFrame
df_prototipo = pd.read_excel(caminho_arquivo)

print(f"Base carregada com sucesso. Respostas processadas: {len(df_prototipo)}")

# Exibe as primeiras linhas para validação da estrutura
display(df_prototipo.head(3))

In [0]:
# ==============================================================================
# BLOCO 3: PROCESSAMENTO DA SYSTEM USABILITY SCALE (SUS)
# ==============================================================================

perguntas_sus = [
    "Eu acho que gostaria de usar esse sistema frequentemente",
    "Eu achei esse sistema desnecessariamente complexo",
    "Eu achei esse sistema fácil de usar",
    "Eu achei que precisaria de ajuda de uma pessoa técnica para ser capaz de usar esse sistema",
    "Eu achei que as várias funções desse sistema foram bem integradas",
    "Eu acho que o sistema apresenta muita inconsistência",
    "Eu imagino que a maioria das pessoas pode aprender a usar esse sistema rapidamente",
    "Eu achei esse sistema muito pesado para usar",
    "Eu me senti muito seguro usando o sistema",
    "Eu precisei aprender muitas coisas antes que pudesse utilizar esse sistema"
]

def calcular_sus_individual(linha):
    soma = 0
    for i, pergunta in enumerate(perguntas_sus):
        colunas_encontradas = [c for c in df_prototipo.columns if pergunta in c]
        if colunas_encontradas:
            resposta = float(linha[colunas_encontradas[0]])
            if (i + 1) % 2 != 0: 
                soma += (resposta - 1)
            else:                
                soma += (5 - resposta)
    return soma * 2.5

df_prototipo['Score_SUS'] = df_prototipo.apply(calcular_sus_individual, axis=1)

media_sus = df_prototipo['Score_SUS'].mean()
desvio_padrao_sus = df_prototipo['Score_SUS'].std()
minimo_sus = df_prototipo['Score_SUS'].min()
maximo_sus = df_prototipo['Score_SUS'].max()

if media_sus >= 85: 
    classificacao = "Excelente"
elif media_sus >= 73: 
    classificacao = "Bom"
elif media_sus >= 50: 
    classificacao = "Ok (Aceitável)"
else: 
    classificacao = "Pobre (Inaceitável)"

tabela_sus = pd.DataFrame({
    "Métrica": ["Score Médio Global", "Classificação", "Desvio Padrão", "Mínimo", "Máximo"],
    "Resultado": [f"{media_sus:.1f}", classificacao, f"{desvio_padrao_sus:.1f}", f"{minimo_sus:.1f}", f"{maximo_sus:.1f}"]
})

print("TABELA RESULTADO DA AVALIAÇÃO DE USABILIDADE DO PROTÓTIPO (SUS)")
print(f"Score Médio Global apurado em {media_sus:.1f} ({classificacao})\n")

spark.createDataFrame(tabela_sus).display()

In [0]:
# ==============================================================================
# BLOCO 4: PROCESSAMENTO DA CARGA COGNITIVA (RAW NASA-TLX)
# ==============================================================================

colunas_nasa = [col for col in df_prototipo.columns if '(0 a 10)' in col]

dimensoes_nasa = [
    "Demanda Mental",
    "Demanda Física",
    "Demanda Temporal",
    "Desempenho Percebido",
    "Esforço Global",
    "Nível de Frustração"
]

df_nasa_num = pd.DataFrame()
medias_nasa = []
desvios_nasa = []

for i, col in enumerate(colunas_nasa):
    serie = pd.to_numeric(df_prototipo[col], errors='coerce')
    medias_nasa.append(serie.mean())
    desvios_nasa.append(serie.std())
    
    # Inverte Desempenho Percebido (4ª dimensão, índice 3) para compor a carga global
    if i == 3:
        df_nasa_num[dimensoes_nasa[i]] = 10 - serie
    else:
        df_nasa_num[dimensoes_nasa[i]] = serie

df_nasa = pd.DataFrame({
    "Dimensão Cognitiva": dimensoes_nasa,
    "Pontuação Média (0 a 10)": [round(m, 1) for m in medias_nasa],
    "Desvio Padrão": [round(d, 1) for d in desvios_nasa]
})

# Raw NASA-TLX: média por participante com Desempenho invertido
carga_por_participante = df_nasa_num.mean(axis=1)
carga_global = carga_por_participante.mean()
dp_carga_global = carga_por_participante.std()

print("TABELA RESULTADO DA AVALIAÇÃO DE CARGA COGNITIVA (RAW NASA-TLX)")
print(f"Índice Global de Carga Cognitiva (com inversão de Desempenho): {carga_global:.1f} (DP = {dp_carga_global:.1f})\n")

spark.createDataFrame(df_nasa).display()

In [0]:
# ==============================================================================
# BLOCO 5: ANÁLISE LEXICAL E FREQUÊNCIA DE N-GRAMAS (DADOS QUALITATIVOS)
# ==============================================================================

# 1. Captura segura e explícita das 5 perguntas abertas (evita pegar colunas calculadas como Score_SUS)
padroes_perguntas_abertas = [
    "maior aplicabilidade",
    "decisões institucionais",
    "limitações podem comprometer",
    "dados adicionais",
    "aprimoramento do modelo"
]

colunas_qualitativas = [
    col for col in df_prototipo.columns
    if any(padrao in str(col).lower() for padrao in padroes_perguntas_abertas)
]

print(f"Colunas qualitativas identificadas ({len(colunas_qualitativas)} de 5):")
for idx, col in enumerate(colunas_qualitativas, start=1):
    n_validas = df_prototipo[col].dropna().astype(str).str.strip().replace("", np.nan).dropna().shape[0]
    print(f"  Q{idx}: {col} -> {n_validas} respostas válidas")

if len(colunas_qualitativas) != 5:
    print("\n[ALERTA] Esperava-se 5 colunas qualitativas. Verifique os nomes das colunas no Excel!")

# 2. Consolidação e limpeza do corpus textual
corpus = []
for col in colunas_qualitativas:
    respostas = df_prototipo[col].dropna().astype(str).tolist()
    corpus.extend(respostas)

def limpar_texto(texto):
    texto = str(texto).lower()
    # Substitui pontuação por espaço para evitar junção indevida de palavras com hífen ou barra
    texto = re.sub(r'[^\w\s]', ' ', texto)
    # Remove números
    texto = re.sub(r'\d+', ' ', texto)
    # Normaliza múltiplos espaços
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

corpus_limpo = []
for doc in corpus:
    doc_limpo = limpar_texto(doc)
    if len(doc_limpo) > 0:
        corpus_limpo.append(doc_limpo)

print(f"\nTotal de respostas consolidadas no corpus limpo: {len(corpus_limpo)}\n")

# 3. Dicionário de exclusão de ruídos (stopwords + termos genéricos das perguntas + ruídos verbais)
stopwords_pt = [
    "a", "o", "as", "os", "e", "de", "do", "da", "dos", "das", "em", "no", "na",
    "nos", "nas", "para", "por", "com", "sem", "um", "uma", "uns", "umas", "que",
    "se", "sua", "seu", "suas", "seus", "como", "mais", "mas", "ou", "ao", "aos",
    "isso", "esse", "essa", "estes", "estas", "aquilo", "este", "esta", "pelo",
    "pela", "pelos", "pelas", "são", "foi", "ser", "ter", "tem", "não", "sim",
    "sobre", "entre", "quando", "muito", "também", "já", "até", "mesmo", "só",
    "fazer", "sendo", "está", "ainda", "qual", "quais", "meio", "podem", "poder",
    "seriam", "seria", "pode", "haver", "tudo", "seja", "deverá", "pessoas",
    "modelo", "sistema", "protótipo", "artefato", "solução", "ferramenta",
    "aplicabilidade", "utilização", "contexto", "real", "decisões", "dados",
    "sugestões", "melhor", "necessários", "ampliar", "utilidade", "acredito"
]

# 4. Processamento da frequência de expressões (Unigramas e Bigramas)
if len(corpus_limpo) > 0:
    vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words=stopwords_pt)
    X = vectorizer.fit_transform(corpus_limpo)

    frequencias = zip(vectorizer.get_feature_names_out(), X.toarray().sum(axis=0))
    df_ngrams = pd.DataFrame(frequencias, columns=["Expressão (N-grama)", "Frequência Absoluta"])

    # Ordenação decrescente (em caso de empate, ordem alfabética garante reprodutibilidade)
    df_ngrams = (
        df_ngrams
        .sort_values(by=["Frequência Absoluta", "Expressão (N-grama)"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # Filtro: termos com frequência absoluta >= 2
    df_ngrams = df_ngrams[df_ngrams["Frequência Absoluta"] >= 2].reset_index(drop=True)

    print("RESULTADO DA ANÁLISE LEXICAL DOS DADOS QUALITATIVOS (BASE PARA A FIGURA 29)")
    print("Frequência de agrupamentos semânticos nas respostas dos especialistas (Q1 a Q5)\n")

    spark.createDataFrame(df_ngrams).display()
else:
    print("Nenhum texto válido encontrado para análise nas colunas mapeadas.")

In [0]:
# ==============================================================================
# BLOCO 5.1: GERAÇÃO DO GRÁFICO DE ANÁLISE LEXICAL COM PLOTLY
# ==============================================================================


# 1. Filtro de limpeza final no DataFrame gerado anteriormente
ruidos_residuais = ['haver', 'tudo', 'seja', 'deverá haver', 'pessoas']
df_plot = df_ngrams[~df_ngrams['Expressão (N-grama)'].isin(ruidos_residuais)].copy()

# Seleciona apenas os 10 termos mais frequentes
df_plot = df_plot.head(10)

# Capitaliza a primeira letra
df_plot['Expressão (N-grama)'] = df_plot['Expressão (N-grama)'].str.capitalize()

# 2. Criação do gráfico interativo com Plotly
fig = px.bar(
    df_plot,
    x='Frequência Absoluta', 
    y='Expressão (N-grama)', 
    orientation='h',
    text='Frequência Absoluta', # Adiciona o número no final da barra
    title='Frequência Lexical nas Respostas (Top Termos)'
)

# 3. Customização das cores e tooltip (hover)
fig.update_traces(
    marker_color='#1f497d', # Azul acadêmico
    textposition='outside', # Número posicionado fora da barra
    textfont=dict(color='black', size=13), # CORREÇÃO: Fonte preta para garantir visibilidade no fundo transparente/branco
    hovertemplate='<b>Termo:</b> %{y}<br><b>Total de Citações:</b> %{x}<extra></extra>',
    cliponaxis=False # Garante que o número não seja cortado pela margem direita
)

# 4. Configuração de Layout
fig.update_layout(
    title_font_size=16,
    margin=dict(t=50, b=40, l=150, r=40), # Margem direita (r=40) dá espaço para o número
    xaxis=dict(title="", showticklabels=False), # Oculta eixo X para visualização mais limpa
    yaxis=dict(title="", categoryorder='total ascending'), # Ordena as barras visualmente
    plot_bgcolor='rgba(0,0,0,0)', # Fundo transparente
    paper_bgcolor='rgba(0,0,0,0)'
)

# 5. Exibição do gráfico
fig.show()

In [0]:
# ==============================================================================
# BLOCO 5.2: MAPA DE ÁRVORE TEMÁTICO (TREEMAP)
# ==============================================================================



# 1. Recuperação e Limpeza dos Dados (Caso a célula anterior tenha sido reiniciada)
ruidos_residuais = ['haver', 'tudo', 'seja', 'deverá haver', 'pessoas']
df_plot = df_ngrams[~df_ngrams['Expressão (N-grama)'].isin(ruidos_residuais)].copy().head(10)
df_plot['Expressão (N-grama)'] = df_plot['Expressão (N-grama)'].str.capitalize()

# 2. Codificação Qualitativa (Mapeamento Temático)
# Atribui cada termo bruto a uma macro-dimensão teórica
def classificar_dimensao(termo):
    termo = str(termo).lower()
    assistencial = ['atendimento', 'rede', 'tea', 'prestadores']
    regulatorio = ['atualização', 'prazos', 'limitação']
    
    if termo in assistencial:
        return 'Suficiência Assistencial'
    elif termo in regulatorio:
        return 'Risco Regulatório'
    else:
        return 'Governança e Processos'

df_plot['Dimensão Teórica'] = df_plot['Expressão (N-grama)'].apply(classificar_dimensao)

# 3. Geração do Treemap Interativo
fig = px.treemap(
    df_plot,
    path=[px.Constant('Eixos de Necessidade'), 'Dimensão Teórica', 'Expressão (N-grama)'],
    values='Frequência Absoluta',
    color='Dimensão Teórica',
    color_discrete_map={
        'Suficiência Assistencial': '#1f497d', # Azul Escuro (Área Core)
        'Governança e Processos': '#4f81bd',   # Azul Médio (Operação)
        'Risco Regulatório': '#a6d8f0',        # Azul Claro (Defesa)
        '(?)': '#e0e0e0'
    }
)

# 4. Customização Visual
fig.update_traces(
    textinfo="label+value",
    textfont=dict(size=14, color='white'),
    hovertemplate='<b>%{label}</b><br>Ocorrências: %{value}<extra></extra>'
)

fig.update_layout(
    title=dict(
        text='Distribuição Lexical por Dimensão Analítica (Treemap)',
        font=dict(size=16)
    ),
    margin=dict(t=50, l=10, r=10, b=10),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

# 5. Exibição do Gráfico
fig.show()